# RAG Chatbot Implementation

A complete, standalone RAG chatbot built from scratch.

**Goal:** Load a PDF, build a retrieval system, and create a working chatbot.

---

## Step 1: Import Libraries and Initialize Client

In [1]:
from openai import OpenAI
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
import os

# Initialize OpenAI client
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

print("✓ Libraries imported")
print("✓ OpenAI client initialized")

✓ Libraries imported
✓ OpenAI client initialized


## Step 2: Load and Extract Text from PDF

In [2]:
# Load PDF
reader = PdfReader("employee_policy_handbook.pdf")

# Extract text from all pages
full_text = ""
for page in reader.pages:
    full_text += page.extract_text()

print(f"✓ PDF loaded")
print(f"✓ Total characters: {len(full_text)}")
print(f"✓ Total pages: {len(reader.pages)}")

✓ PDF loaded
✓ Total characters: 3165
✓ Total pages: 3


## Step 3: Chunk the Document

In [3]:
# Create recursive chunker
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

# Split text into chunks
chunks = splitter.split_text(full_text)

print(f"✓ Document chunked")
print(f"✓ Total chunks: {len(chunks)}")
print(f"\nFirst chunk:")
print(chunks[0])

✓ Document chunked
✓ Total chunks: 13

First chunk:
Employee  Policy  Handbook  
Section  1:  Employment  Categories  
The  company  has  three  employment  categories:  
1.  Full-Time  Employee  2.  Intern  3.  Contract  Employee  
Each  category  follows  different  employment  rules  and  exit  procedures.  
 
Section  2:  Notice  Period  Policy


## Step 4: Embed All Chunks

In [4]:
# Embed all chunks
print("Embedding chunks...")

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunks
)

chunk_embeddings = [
    item.embedding
    for item in response.data
]

print(f"✓ Embeddings created")
print(f"✓ Total embeddings: {len(chunk_embeddings)}")
print(f"✓ Embedding dimensions: {len(chunk_embeddings[0])}")

Embedding chunks...
✓ Embeddings created
✓ Total embeddings: 13
✓ Embedding dimensions: 1536


In [18]:
chunk_embeddings[0]==chunk_embeddings[0]

True

# Additional Step
# Vector Store DB

---

## RAG Chatbot Functions

Now we have:
- Chunks from the PDF
- Embeddings for each chunk

Next: Build the retrieval and generation functions.

### Function 1: Retrieve Similar Chunks

In [9]:
def retrieve_chunks(question, top_k=3):
    """
    Retrieve the top-K most relevant chunks for a question.
    
    Args:
        question: User's question (string)
        top_k: Number of chunks to retrieve (default: 3)
    
    Returns:
        List of (chunk, similarity_score) tuples, sorted by score
    """
    import warnings
    
    # Suppress numerical warnings from scikit-learn
    warnings.filterwarnings('ignore', category=RuntimeWarning)
    
    # Step 1: Embed the question
    query_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    )
    query_embedding = query_response.data[0].embedding
    
    # Step 2: Calculate similarity with each chunk
    scores = []
    for chunk, embedding in zip(chunks, chunk_embeddings):
        score = cosine_similarity(
            [query_embedding],
            [embedding]
        )[0][0]
        scores.append((chunk, score))
    
    # Step 3: Sort by similarity and return top-K
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

print("✓ retrieve_chunks() function defined")

✓ retrieve_chunks() function defined


### Function 2: Main Chatbot - Ask PDF

In [10]:
def ask_pdf(question, top_k=3):
    """
    Ask the PDF chatbot a question and get a grounded answer.
    
    The RAG Pipeline:
    1. Retrieve relevant chunks
    2. Build context from chunks
    3. Create prompt with context
    4. Call LLM to generate answer
    
    Args:
        question: User's question (string)
        top_k: Number of chunks to use as context (default: 3)
    
    Returns:
        Answer string from the LLM
    """
    # Step 1: Retrieve relevant chunks
    retrieved = retrieve_chunks(question, top_k=top_k)
    
    # Step 2: Build context from retrieved chunks
    context = "\n\n".join(
        chunk for chunk, score in retrieved
    )
    
    # Step 3: Build the prompt
    prompt = f"""
You are an HR policy assistant.

Answer ONLY using the provided context.

If the answer cannot be found in the context,
respond with: Not found in context.

Context:
{context}

Question:
{question}
"""
    
    # Step 4: Call LLM to generate answer
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    return response.choices[0].message.content

print("✓ ask_pdf() function defined")

✓ ask_pdf() function defined


---

## Using the Chatbot

The RAG chatbot is ready! Simply call `ask_pdf(question)` with any question.

### Question 1: Notice Period

In [11]:
question = "What is the notice period for interns?"
answer = ask_pdf(question)

print(f"Question: {question}")
print(f"\nAnswer: {answer}")

Question: What is the notice period for interns?

Answer: Interns must serve a 15-day notice period.


### Question 2: Health Insurance

In [12]:
question = "Do interns receive health insurance?"
answer = ask_pdf(question)

print(f"Question: {question}")
print(f"\nAnswer: {answer}")

Question: Do interns receive health insurance?

Answer: Not found in context.


### Question 3: Laptop Return

In [13]:
question = "When should interns return laptops?"
answer = ask_pdf(question)

print(f"Question: {question}")
print(f"\nAnswer: {answer}")

Question: When should interns return laptops?

Answer: Interns must return laptops at least 2 days before their exit date.


### Question 4: Work from Home Policy

In [14]:
question = "What is the work from home policy?"
answer = ask_pdf(question)

print(f"Question: {question}")
print(f"\nAnswer: {answer}")

Question: What is the work from home policy?

Answer: Not found in context.


### Question 5: Exit Requirements

In [15]:
question = "What must interns do before leaving?"
answer = ask_pdf(question)

print(f"Question: {question}")
print(f"\nAnswer: {answer}")

Question: What must interns do before leaving?

Answer: Interns must complete the following steps before leaving: inform their manager before submitting a formal resignation request, serve a 15-day notice period, and ensure that any emergency leave requests are approved by the reporting manager.


---

## Try Your Own Questions

Use the cell below to ask any question about the employee handbook:

In [16]:
# Ask your own question
my_question = "Your question here?"
my_answer = ask_pdf(my_question)

print(f"Question: {my_question}")
print(f"\nAnswer: {my_answer}")

Question: Your question here?

Answer: Not found in context.


---

## How It Works

### The RAG Pipeline

```
┌─────────────────────────────────────────┐
│          RAG Chatbot Pipeline            │
├─────────────────────────────────────────┤
│                                          │
│  1. User Question                        │
│         ↓                                │
│  2. Embed Question                       │
│         ↓                                │
│  3. Search Similar Chunks (cosine)       │
│         ↓                                │
│  4. Retrieve Top 3 Chunks                │
│         ↓                                │
│  5. Build Context                        │
│         ↓                                │
│  6. Create Prompt                        │
│         ↓                                │
│  7. Call GPT-4o-mini                     │
│         ↓                                │
│  8. Return Grounded Answer               │
│                                          │
└─────────────────────────────────────────┘
```

### Key Components

**retrieve_chunks(question, top_k=3)**
- Embeds the question
- Compares against all chunk embeddings
- Returns top-K most similar chunks with scores

**ask_pdf(question, top_k=3)**
- Calls retrieve_chunks()
- Builds context from retrieved chunks
- Sends context + question to LLM
- Returns grounded answer

### Why This Works

1. **Chunking** divides the PDF into manageable pieces
2. **Embeddings** convert chunks into semantic vectors
3. **Similarity Search** finds relevant chunks for any question
4. **Context Building** gives the LLM relevant information
5. **Generation** creates accurate, grounded answers

The result: A chatbot that can answer questions about the PDF accurately and reliably.